# Halo Profile Validation

**Part 2 of BCM Degeneracy Analysis**

This notebook validates that Replace profiles are correctly constructed as mixtures of DMO and Hydro profiles based on the replacement region (mass range and radial extent).

Key insight: Replace profiles are a **mixture** of DMO and Hydro:
- Inside replacement region: profile = Hydro
- Outside replacement region: profile = DMO

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import h5py
import warnings
warnings.filterwarnings('ignore')

try:
    import scienceplots
    plt.style.use(['science', 'notebook'])
except:
    pass

print('Libraries loaded successfully')

## 1. Load Profile Data

In [ ]:
PROFILE_PATH = '/mnt/home/mlee1/ceph/hydro_replace_fields/L205n2500TNG/profiles/profiles_snap096.h5'

with h5py.File(PROFILE_PATH, 'r') as f:
    print('Available datasets:', list(f.keys()))
    
    # Load from attributes
    mass_bin_edges = f.attrs['mass_bin_edges']
    radial_bin_edges = f.attrs['radial_bins']
    
    # Compute bin centers
    mass_bins = mass_bin_edges  # Edges: [12, 12.5, 13, 13.5, 14, 14.5, 16]
    r_centers = np.sqrt(radial_bin_edges[:-1] * radial_bin_edges[1:])  # Geometric mean
    
    # Load stacked profiles
    rho_hydro = f['stacked_hydro'][:]
    rho_dmo = f['stacked_dmo'][:]
    counts_hydro = f['counts_hydro'][:]
    counts_dmo = f['counts_dmo'][:]

# Get total halo counts per mass bin (sum over radial bins, take first radial bin as representative)
counts = counts_hydro[:, 0]

print(f'\nMass bin edges (log10 M/Msun): {mass_bins}')
print(f'Radial range: {r_centers.min():.3f} - {r_centers.max():.3f} R200')
print(f'Number of radial bins: {len(r_centers)}')
print(f'Profile shapes: hydro={rho_hydro.shape}, dmo={rho_dmo.shape}')
print(f'Halo counts per mass bin: {counts}')

## 2. Define Profile Construction Functions

In [ ]:
def construct_replace_profile(rho_hydro, rho_dmo, r_centers, mass_idx, 
                               r_inner, r_outer, m_lower, m_upper, mass_bins):
    """
    Construct what a Replace model's profile would look like.
    
    For halos IN the mass range [m_lower, m_upper]:
        - r < r_inner: DMO (not replaced)
        - r_inner <= r <= r_outer: Hydro (replaced)
        - r > r_outer: DMO (not replaced)
    
    For halos OUTSIDE mass range: all DMO
    """
    # Use the lower edge of the mass bin
    log_m = mass_bins[mass_idx]
    
    # Check if this mass bin is within replacement range
    if m_lower <= log_m < m_upper:
        # Create mixed profile
        profile = rho_dmo[mass_idx].copy()
        mask = (r_centers >= r_inner) & (r_centers <= r_outer)
        profile[mask] = rho_hydro[mass_idx, mask]
        return profile
    else:
        return rho_dmo[mass_idx].copy()


def profile_mismatch(rho_replace, rho_hydro, r_centers):
    """
    Compute weighted profile error.
    
    epsilon = sqrt(sum(|delta_rho/rho|^2 * r^2) / sum(r^2))
    
    Weight by r^2 to emphasize physically important outer regions.
    """
    valid = rho_hydro > 0
    if not np.any(valid):
        return np.nan
    
    delta_rho = np.zeros_like(rho_replace)
    delta_rho[valid] = (rho_replace[valid] - rho_hydro[valid]) / rho_hydro[valid]
    
    weights = r_centers**2
    epsilon = np.sqrt(np.sum(delta_rho**2 * weights) / np.sum(weights))
    return epsilon


print('Profile construction functions defined')

## 3. Define Scenarios

In [ ]:
# Define the three scenarios to compare
SCENARIOS = {
    'A: Outer shells\n(3-5 R200, all M)': {
        'r_inner': 3.0, 'r_outer': 5.0, 
        'm_lower': 12.0, 'm_upper': 16.0,
        'color': 'C0', 'short': 'A'
    },
    'B: Massive cores\n(0-1 R200, M>10^13)': {
        'r_inner': 0.0, 'r_outer': 1.0, 
        'm_lower': 13.0, 'm_upper': 16.0,
        'color': 'C1', 'short': 'B'
    },
    'C: Complete\n(0-5 R200, M>10^12)': {
        'r_inner': 0.0, 'r_outer': 5.0, 
        'm_lower': 12.0, 'm_upper': 16.0,
        'color': 'C2', 'short': 'C'
    }
}

# Mass bins to analyze (indices 0-3 = 10^12-12.5, 10^12.5-13, 10^13-13.5, 10^13.5-14)
MASS_INDICES = [0, 1, 2, 3]

print('Scenarios defined:')
for name, params in SCENARIOS.items():
    print(f"  {params['short']}: r=[{params['r_inner']}-{params['r_outer']}] R200, "
          f"M=[10^{params['m_lower']}-10^{params['m_upper']}]")
print(f'\nMass bins to analyze: {[f"10^{mass_bins[i]:.1f}" for i in MASS_INDICES]}')

## 4. Profile Comparison Plot (4x3 Grid)

In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(12, 14), sharex=True)

for row, mass_idx in enumerate(MASS_INDICES):
    mass_label = f'$10^{{{mass_bins[mass_idx]:.1f}}}-10^{{{mass_bins[mass_idx+1]:.1f}}}$ M$_\\odot$'
    
    for col, (scenario_name, params) in enumerate(SCENARIOS.items()):
        ax = axes[row, col]
        
        # Construct Replace profile for this scenario
        rho_replace = construct_replace_profile(
            rho_hydro, rho_dmo, r_centers, mass_idx,
            params['r_inner'], params['r_outer'],
            params['m_lower'], params['m_upper'], mass_bins
        )
        
        # Compute ratio to Hydro
        valid = rho_hydro[mass_idx] > 0
        ratio = np.ones_like(r_centers)
        ratio[valid] = rho_replace[valid] / rho_hydro[mass_idx, valid]
        
        # Compute mismatch
        epsilon = profile_mismatch(rho_replace, rho_hydro[mass_idx], r_centers)
        
        # Plot
        ax.axhline(1, color='gray', ls='--', alpha=0.5, lw=1)
        ax.fill_between(r_centers, 0.9, 1.1, color='green', alpha=0.15, label=r'$\pm$10%')
        
        # Show replacement region
        log_m = mass_bins[mass_idx]
        if params['m_lower'] <= log_m < params['m_upper']:
            ax.axvspan(params['r_inner'], params['r_outer'], 
                      color=params['color'], alpha=0.2, label='Replaced')
        
        ax.semilogx(r_centers, ratio, color=params['color'], lw=2)
        
        # Annotate
        ax.text(0.95, 0.05, f'$\\varepsilon = {epsilon:.3f}$', 
               transform=ax.transAxes, ha='right', va='bottom',
               fontsize=10, bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
        
        ax.set_xlim(0.01, 5)
        ax.set_ylim(0.5, 1.5)
        
        # Labels
        if row == 0:
            ax.set_title(scenario_name, fontsize=11)
        if col == 0:
            ax.set_ylabel(f'{mass_label}\n' + r'$\rho_{\rm Replace}/\rho_{\rm Hydro}$', fontsize=10)
        if row == 3:
            ax.set_xlabel(r'$r/R_{200}$')
        
        # Add halo count
        if col == 2:
            ax.text(1.02, 0.5, f'N={int(counts[mass_idx])}', 
                   transform=ax.transAxes, rotation=-90, va='center', fontsize=9)

# Add legend to first panel
axes[0, 0].legend(loc='upper left', fontsize=8)

plt.tight_layout()
plt.savefig('figures/halo_profile_validation.png', dpi=150, bbox_inches='tight')
print('Saved figures/halo_profile_validation.png')
plt.show()

## 5. Profile Error Heatmap

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

n_mass = len(MASS_INDICES)
n_r = len(r_centers)

for ax_idx, (scenario_name, params) in enumerate(SCENARIOS.items()):
    ax = axes[ax_idx]
    
    # Build error matrix
    error_matrix = np.zeros((n_mass, n_r))
    
    for i, mass_idx in enumerate(MASS_INDICES):
        rho_replace = construct_replace_profile(
            rho_hydro, rho_dmo, r_centers, mass_idx,
            params['r_inner'], params['r_outer'],
            params['m_lower'], params['m_upper'], mass_bins
        )
        
        valid = rho_hydro[mass_idx] > 0
        error_matrix[i, valid] = np.abs(
            (rho_replace[valid] - rho_hydro[mass_idx, valid]) / rho_hydro[mass_idx, valid]
        )
    
    # Plot heatmap
    im = ax.pcolormesh(r_centers, np.arange(n_mass + 1), error_matrix, 
                       cmap='RdYlGn_r', vmin=0, vmax=0.5, shading='auto')
    
    # Mark replacement region boundary
    if params['r_inner'] > 0:
        ax.axvline(params['r_inner'], color='white', ls='--', lw=2, alpha=0.8)
    ax.axvline(params['r_outer'], color='white', ls='--', lw=2, alpha=0.8)
    
    # Mark which mass bins are replaced
    for i, mass_idx in enumerate(MASS_INDICES):
        log_m = mass_bins[mass_idx]
        if log_m >= params['m_lower'] and log_m < params['m_upper']:
            ax.axhspan(i, i+1, xmin=0, xmax=0.03, color='blue', alpha=0.7)
    
    ax.set_xlabel(r'$r/R_{200}$')
    ax.set_xscale('log')
    ax.set_xlim(0.01, 5)
    
    if ax_idx == 0:
        ax.set_ylabel(r'Mass bin')
    
    ax.set_title(f"Scenario {params['short']}: {scenario_name.split(chr(10))[1]}")
    
    # Y-axis labels
    ax.set_yticks(np.arange(n_mass) + 0.5)
    ax.set_yticklabels([f'$10^{{{mass_bins[m]:.1f}}}$' for m in MASS_INDICES])
    
    plt.colorbar(im, ax=ax, label=r'$|\Delta\rho/\rho_{\rm Hydro}|$')

plt.tight_layout()
plt.savefig('figures/profile_error_heatmap.png', dpi=150, bbox_inches='tight')
print('Saved figures/profile_error_heatmap.png')
plt.show()

## 6. Summary Table

In [ ]:
print('=' * 70)
print('Profile Mismatch Summary (epsilon = weighted RMS relative error)')
print('=' * 70)
print(f'{"Mass Bin":^25} | {"eps_A":^10} | {"eps_B":^10} | {"eps_C":^10}')
print('-' * 70)

epsilons = {name: [] for name in SCENARIOS.keys()}

for mass_idx in MASS_INDICES:
    mass_label = f'10^{mass_bins[mass_idx]:.1f} - 10^{mass_bins[mass_idx+1]:.1f}'
    
    row_values = []
    for scenario_name, params in SCENARIOS.items():
        rho_replace = construct_replace_profile(
            rho_hydro, rho_dmo, r_centers, mass_idx,
            params['r_inner'], params['r_outer'],
            params['m_lower'], params['m_upper'], mass_bins
        )
        eps = profile_mismatch(rho_replace, rho_hydro[mass_idx], r_centers)
        epsilons[scenario_name].append(eps)
        row_values.append(eps)
    
    print(f'{mass_label:^25} | {row_values[0]:^10.4f} | {row_values[1]:^10.4f} | {row_values[2]:^10.4f}')

print('-' * 70)

# Mean epsilon
mean_eps = [np.mean(epsilons[name]) for name in SCENARIOS.keys()]
print(f'{"Mean":^25} | {mean_eps[0]:^10.4f} | {mean_eps[1]:^10.4f} | {mean_eps[2]:^10.4f}')
print('=' * 70)

print('\nInterpretation:')
print('  eps = 0: Perfect match to Hydro (complete replacement)')
print('  eps > 0: Deviation from Hydro due to incomplete replacement')
print('  Scenario C should have eps ~ 0 everywhere (full replacement)')

## 7. Radial Profile Comparison (Single Mass Bin)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Choose mass bin 10^13 - 10^13.5
mass_idx = 2
mass_label = f'$10^{{{mass_bins[mass_idx]:.1f}}}-10^{{{mass_bins[mass_idx+1]:.1f}}}$ M$_\\odot$/h'

# Left panel: Absolute profiles
ax = axes[0]
ax.loglog(r_centers, rho_dmo[mass_idx], 'k--', lw=2, label='DMO')
ax.loglog(r_centers, rho_hydro[mass_idx], 'k-', lw=2, label='Hydro')

for scenario_name, params in SCENARIOS.items():
    rho_replace = construct_replace_profile(
        rho_hydro, rho_dmo, r_centers, mass_idx,
        params['r_inner'], params['r_outer'],
        params['m_lower'], params['m_upper'], mass_bins
    )
    ax.loglog(r_centers, rho_replace, ls=':', lw=2, 
             color=params['color'], label=f"Replace {params['short']}")

ax.set_xlabel(r'$r/R_{200}$')
ax.set_ylabel(r'$\rho$ [stacked profile units]')
ax.set_title(f'Stacked Density Profiles ({mass_label})')
ax.legend(loc='upper right', fontsize=9)
ax.set_xlim(0.01, 5)

# Right panel: Hydro/DMO ratio
ax = axes[1]
valid = rho_dmo[mass_idx] > 0
ratio_hydro_dmo = np.ones_like(r_centers)
ratio_hydro_dmo[valid] = rho_hydro[mass_idx, valid] / rho_dmo[mass_idx, valid]

ax.axhline(1, color='gray', ls='--', alpha=0.5)
ax.semilogx(r_centers, ratio_hydro_dmo, 'k-', lw=2, label='Hydro/DMO')

for scenario_name, params in SCENARIOS.items():
    rho_replace = construct_replace_profile(
        rho_hydro, rho_dmo, r_centers, mass_idx,
        params['r_inner'], params['r_outer'],
        params['m_lower'], params['m_upper'], mass_bins
    )
    ratio = np.ones_like(r_centers)
    ratio[valid] = rho_replace[valid] / rho_dmo[mass_idx, valid]
    ax.semilogx(r_centers, ratio, ls=':', lw=2, 
               color=params['color'], label=f"Replace {params['short']}/DMO")

ax.set_xlabel(r'$r/R_{200}$')
ax.set_ylabel(r'$\rho/\rho_{\rm DMO}$')
ax.set_title(f'Ratio to DMO ({mass_label})')
ax.legend(loc='lower right', fontsize=9)
ax.set_xlim(0.01, 5)
ax.set_ylim(0.5, 1.5)

plt.tight_layout()
plt.savefig('figures/profile_comparison_single_mass.png', dpi=150, bbox_inches='tight')
print('Saved figures/profile_comparison_single_mass.png')
plt.show()

## Summary

This validation confirms:

1. **Scenario A (Outer shells)**: Only replaces r > 3 R200, leaving cores DMO -> large error in inner regions
2. **Scenario B (Massive cores)**: Only replaces M > 10^13 halos -> lower mass bins are pure DMO  
3. **Scenario C (Complete)**: Full replacement -> epsilon ~ 0 everywhere

The profile mismatch metric epsilon quantifies how well each Replace model approximates the full Hydro profile.